# 03 — デモ: Session 3a 不整地 random_boxes

**目的:** 足場最適化 ON + 段差 terrain で **MPCが着地点も計画** する様子を理解する。

**難度:** 中（perlin より先に試す）


## 4セッションの違い（必読）

| | **S1 本Notebook** | S2 tune | **S3a boxes ← 今ここ** | S3b perlin |
|---|-------------------|---------|-----------|------------|
| **scene** | **flat（平坦）** | flat | **random_boxes（箱）** | **perlin（連続起伏）** |
| **足場最適化** | **OFF** | OFF | ON | ON |
| **主な目的** | 最小構成で動作確認 | μ / 歩調チューニング | 段差・離散障害 | 連続起伏 |
| **デモGIFで見る点** | 平坦＋標準trot | 平坦＋**速いtrot** | **箱が見える** | **うねり地形** |
| **GIF** | demo_s01_flat | demo_s02_tune | demo_s03_boxes | demo_s03_perlin |

> **S1 と S2 は地形とも平坦**です。GIFの違いは **歩調（S2は step_freq=1.75 Hz の速い trot）** と **Notebook内の実験内容** です。  
> **S3a/S3b のデモ GIF は 10 秒再生**し、箱/起伏エリア（x≈6 m 以降）まで走行してキャプチャしています。


### このセッション固有のポイント

- **地形:** `scene=random_boxes` — x≈1 m 以降に **MuJoCo の箱ジオメトリ** が並ぶ（平坦スタート→障害物エリアへ進入）
- **足場最適化:** ON — 着地点を MPC が計画
- **デモGIF:** **10 秒再生**＋ワイド intro で箱エリア（x≈6 m 以降）まで進入。左上 `Session 3a | scene=random_boxes`
- **旧GIFが平坦に見えた理由:** 600 step（≈1.2 s）では箱エリア（x≈1 m）に到達する前に終了していた

![Session 3a demo](../assets/demo_s03_boxes.gif)


In [ ]:
import sys
from pathlib import Path

# mpc_dog ルートを sys.path に追加
ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "scripts" / "pympc_lab.py").exists():
        ROOT = p
        break
sys.path.insert(0, str(ROOT / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pympc_lab import (
    TUNING_GUIDE,
    apply_preset,
    compare_runs,
    load_param_study,
    load_preset_yaml,
    plot_friction_cone,
    run_flat_sim,
    run_speed_terrain_sim,
    run_speed_terrain_sim_resilient,
)

from tuning_labs import (
    TUNING_LABS,
    list_labs,
    run_lab,
    run_lab_pair,
    plot_speed_trial_journey,
    plot_param_study_mu,
    load_cached_lab_results,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4)
print(f"repo: {ROOT}")


## Step 1 — プリセット確認

In [ ]:
import yaml
preset = load_preset_yaml("session03_rough_boxes")
print(yaml.dump(preset, allow_unicode=True))


**設計意図**

- `use_foothold_optimization: True` → 不整地の核心  
- `step_freq: 1.2` → 低め（安定優先）  
- `mu: 0.48` → やや保守的


## Step 2 — プリセット適用

In [ ]:
apply_preset("session03_rough_boxes")


## Step 3 — ❌ vs ✅ 足場 opt OFF vs ON 比較

**教訓:** OFF で転倒しやすくても、ON で変な足場 → 地形推定/制約の問題かも。必ず **両方** 見る。


In [ ]:
apply_preset("session03_rough_boxes")
# OFF: 平坦用設定に近い
off = run_flat_sim(seconds=5.0, scene="random_boxes", use_foothold_optimization=False, step_freq=1.4)
# ON: preset 通り
apply_preset("session03_rough_boxes")
on = run_flat_sim(seconds=5.0, scene="random_boxes", use_foothold_optimization=True, step_freq=1.2, mu=0.48)
fig = compare_runs([("FOOTHOLD OFF", off), ("FOOTHOLD ON", on)])
plt.show()
print("OFF terminated:", off["terminated"], "min_z:", f"{off['min_z']:.3f}")
print("ON  terminated:", on["terminated"],  "min_z:", f"{on['min_z']:.3f}")


## Step 4 — ❌ 典型失敗: step_freq 高すぎ

不整地では **速さより安定**。Session 2 の知見を適用。


In [ ]:
apply_preset("session03_rough_boxes")
fast = run_flat_sim(seconds=4.0, scene="random_boxes", step_freq=1.6, use_foothold_optimization=True)
slow = run_flat_sim(seconds=4.0, scene="random_boxes", step_freq=1.1, duty_factor=0.75, use_foothold_optimization=True)
fig = compare_runs([("FAIL freq=1.6", fast), ("OK freq=1.1 duty=0.75", slow)])
plt.show()


## Step 5 — デモ映像

[`../assets/demo_s03_boxes.gif`](../assets/demo_s03_boxes.gif) — 箱地形＋足場 opt ON

---

## Step 6 — MPC設計者チェックリスト

- [ ] 足場 opt ON の **目的** を1文で言える  
- [ ] OFF/ON 比較で何が変わるか説明できる  
- [ ] 不整地では step_freq↓ / duty↑ / mu↓ の **トリアージ** ができる  
- [ ] S1/S2（平坦）との違いを **地形と足場opt** で説明できる  

**次:** [04_demo_session03b_rough_perlin.ipynb](./04_demo_session03b_rough_perlin.ipynb)
